# Busca híbrida em RAG

Compare busca lexical, similaridade de cosseno e Reciprocal Rank Fusion usando consultas rotuladas.

[Artigo completo](../article/ARTICLE.md)

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

cwd = Path.cwd()
lab = next(
    path
    for path in (cwd, cwd.parent, cwd / "0012_busca_hibrida")
    if path.name == "0012_busca_hibrida" and (path / "src" / "main.py").is_file()
)

def run(*options):
    completed = subprocess.run(
        [sys.executable, "-X", "utf8", str(lab / "src" / "main.py"), *options],
        capture_output=True, check=True, text=True, encoding="utf-8",
    )
    return json.loads(completed.stdout)

Execute a próxima célula para reproduzir a comparação e verificar os resultados com asserções.

Com `--top 2`, a fusão encontra os dois documentos pertinentes para `redefinir senha`. Para `protocolo RE42`, a fusão encontra um e a busca vetorial encontra dois. Use `--top 3` para investigar o efeito do corte.

In [ ]:
at_two = run("--top", "2")
at_three = run("--top", "3")
assert at_two[0]["hybrid"]["recall_at_k"] == 1.0
assert at_two[1]["hybrid"]["recall_at_k"] == 0.5
assert at_two[1]["vector"]["recall_at_k"] == 1.0
assert at_three[1]["hybrid"]["recall_at_k"] == 1.0
for record in at_two:
    print(record["query"])
    for method in ("lexical", "vector", "hybrid"):
        result = record[method]
        print(" ", method, result["results"], "Recall@2:", result["recall_at_k"],
              "RR@2:", result["reciprocal_rank_at_k"])

## Refaça as contribuições da fusão no caso RE42

Calcule o escore pela posição em cada lista. Verifique por que o aviso passa na frente do alerta e como o corte 3 muda o recall.

In [ ]:
lexical = at_three[1]["lexical"]["results"]
vector = at_three[1]["vector"]["results"]
contributions = {}
for ranking in (lexical, vector):
    for position, document_id in enumerate(ranking, start=1):
        contributions[document_id] = contributions.get(document_id, 0.0) + 1 / (60 + position)
reconstructed = sorted(contributions, key=lambda item: (-contributions[item], item))[:3]
assert reconstructed == at_three[1]["hybrid"]["results"]
for document_id in reconstructed:
    print(document_id, f"{contributions[document_id]:.6f}")
print("Recall@3:", at_three[1]["hybrid"]["recall_at_k"])